**Text extraction with OCR**

In [ ]:
from pdf2image import convert_from_path
import pytesseract
from PIL import Image
import os

# 1. Convertir le PDF en images (300 dpi recommandé pour l'OCR)
images = convert_from_path("docs/SCASC_notice_logement_locatif.pdf", dpi=300)

# 2. Appliquer l'OCR sur chaque image
texte_total = ""

for i, image in enumerate(images):
    text = pytesseract.image_to_string(image, lang="fra")
    texte_total += f"\n--- Page {i + 1} ---\n{text.strip()}\n"

# 3. Sauvegarder dans un fichier texte
with open("texte_ocr_total.txt", "w", encoding="utf-8") as f:
    f.write(texte_total)

***Sliding Window Chunking***

In [21]:
from transformers import AutoTokenizer
import json

# === CONFIGURATION ===
MODEL_NAME = "camembert-base"  # tokenizer français
CHUNK_SIZE = 500
OVERLAP = 100

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def sliding_window_chunking(text: str, chunk_size=500, overlap=100):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    chunk_id = 1

    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append({
            "chunk_id": chunk_id,
            "contenu": chunk_text.strip()
        })
        start += chunk_size - overlap
        chunk_id += 1

    return chunks


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/811k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

In [22]:
# Charger le texte nettoyé
with open("outputs/texte_ocr_total.txt", "r", encoding="utf-8") as f:
    texte = f.read()

# Appliquer le chunking
chunks = sliding_window_chunking(texte, chunk_size=500, overlap=100)

# Sauvegarder en JSON
with open("outputs/chunks_sliding_window.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"✅ {len(chunks)} chunks générés avec sliding window.")


Token indices sequence length is longer than the specified maximum sequence length for this model (1631 > 512). Running this sequence through the model will result in indexing errors


✅ 5 chunks générés avec sliding window.


***Générer les vecteurs d’embeddings***

In [36]:
#multilingue, rapide, très utilisé
# model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_name = "sentence-transformers/all-mpnet-base-v2"

In [37]:
from sentence_transformers import SentenceTransformer
import json

# Charger les chunks
with open("outputs/chunks_sliding_window.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# Charger le modèle d'embeddings
model = SentenceTransformer(model_name)

# Calcul des embeddings
for chunk in chunks:
    embedding = model.encode(chunk["contenu"])
    chunk["embedding"] = embedding.tolist()  # Convertir numpy array en liste JSON

# Sauvegarder avec embeddings
with open("outputs/chunks_embed.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"✅ Embeddings ajoutés à {len(chunks)} chunks.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings ajoutés à 5 chunks.


***création d’un index FAISS***

In [38]:
import faiss
import numpy as np
import json

# Charger les chunks avec embeddings
with open("outputs/chunks_embed.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

# Récupérer les vecteurs sous forme de matrice numpy
embedding_dim = len(chunks[0]["embedding"])
vectors = np.array([chunk["embedding"] for chunk in chunks]).astype("float32")

# Créer l'index FAISS
index = faiss.IndexFlatL2(embedding_dim)  # L2 = distance euclidienne
index.add(vectors)

# Sauvegarder les métadonnées associées aux vecteurs
metadata = [{"chunk_id": c["chunk_id"], "contenu": c["contenu"]} for c in chunks]

# Sauvegarder index et métadonnées
faiss.write_index(index, "outputs/faiss_index.index")
with open("outputs/faiss_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"✅ Index FAISS créé avec {index.ntotal} vecteurs.")


✅ Index FAISS créé avec 5 vecteurs.


***recherche sémantique dans FAISS***

In [40]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import json

# Charger l'index et le modèle
index = faiss.read_index("outputs/faiss_index.index")
# model = SentenceTransformer(model_name)

# Charger les métadonnées
with open("outputs/faiss_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

# Fonction de recherche
def rechercher_texte(question, top_k=3):
    query_embedding = model.encode([question]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)

    print("🔎 Résultats similaires :")
    for rank, idx in enumerate(indices[0]):
        print(f"\n--- Résultat {rank + 1} ---")
        print(metadata[idx]["contenu"][:500])  # extrait limité


In [41]:
rechercher_texte("Quelles sont les conditions pour bénéficier de l’aide au logement ?")

🔎 Résultats similaires :

--- Résultat 1 ---
--- Page 1 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Notice d'utilisation Accès à un ement locatif Aix Marseille Université --- Page 2 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Prestation sous condition de ressources QF SCASC © 16 000 € En quelques mots... Vous déménagez ou vous accédez à un premier logement ? Selon vos revenus et si vous remplissez l'un des critères, vous pouvez bénéficier d’une aide financière. Vote initial au C

--- Résultat 2 ---
à un logement social Situation à caractère social exceptionnel (sur proposition du travailleur social AMU) L'aide est limitée aux déménagements dans le Grand Sud-Est. --- Page 7 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Montant En fonction du quotient familial SCASC, le montant de l'allocation est le suivant : + QF < 10 000 : 1 100€ + 10 000 < QF < 12 000 : 900€ + 12 000 < QF < 16 000 : 750€ Cas pa

In [42]:
rechercher_texte("Qui peut en bénéficier ?")

🔎 Résultats similaires :

--- Résultat 1 ---
--- Page 1 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Notice d'utilisation Accès à un ement locatif Aix Marseille Université --- Page 2 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Prestation sous condition de ressources QF SCASC © 16 000 € En quelques mots... Vous déménagez ou vous accédez à un premier logement ? Selon vos revenus et si vous remplissez l'un des critères, vous pouvez bénéficier d’une aide financière. Vote initial au C

--- Résultat 2 ---
d'enseignement) Les personnels en instance de mutation/mobilité hors AMU, fin de contrat, démission, licenciement ou détachement ne peuvent pas bénéficier de cette aide. --- Page 5 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Conditions de ressources Pour pouvoir bénéficier de cette prestation, le Quotient Familial SCASC (QF SCASC) annuel du foyer doit être inférieur ou égal à 16 000 € / an. Revenu fi

In [ ]:
rechercher_texte("Pièces justificatives")

🔎 Résultats similaires :

--- Résultat 1 ---
--- Page 1 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Notice d'utilisation Accès à un ement locatif Aix Marseille Université --- Page 2 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Prestation sous condition de ressources QF SCASC © 16 000 € En quelques mots... Vous déménagez ou vous accédez à un premier logement ? Selon vos revenus et si vous remplissez l'un des critères, vous pouvez bénéficier d’une aide financière. Vote initial au C

--- Résultat 2 ---
à un logement social Situation à caractère social exceptionnel (sur proposition du travailleur social AMU) L'aide est limitée aux déménagements dans le Grand Sud-Est. --- Page 7 --- Prestation d'action sociale Accès à un logement locatif Septembre 2024 Montant En fonction du quotient familial SCASC, le montant de l'allocation est le suivant : + QF < 10 000 : 1 100€ + 10 000 < QF < 12 000 : 900€ + 12 000 < QF < 16 000 : 750€ Cas pa

: 